In [8]:
"""
TOP1模型 (Gp_GP_Matern_0.5_K6_Comb17) PDP分析程序 v5
完整包含：1D PDP + 2D交互 + 3D交互
修改：禁用异常值检测（202个样本已是高质量筛选数据）
修复v1：ScaledPredictor 增加 score 方法
修复v2：GPWrapper / ScaledPredictor 增加 __sklearn_is_fitted__ 方法
修复v3：删除 __sklearn_tags__（版本兼容问题）
修复v4：2D PDP 完全手动计算，彻底绕过 partial_dependence 的估计器类型检查
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()

import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
import pickle
import os
from itertools import combinations
import warnings
from scipy import stats

warnings.filterwarnings('ignore')
plt.ioff()
matplotlib.use('Agg')

matplotlib.rcParams['font.family']              = 'Arial'
matplotlib.rcParams['pdf.fonttype']             = 42
matplotlib.rcParams['ps.fonttype']              = 42
matplotlib.rcParams['axes.unicode_minus']       = False
matplotlib.rcParams['figure.dpi']              = 300
matplotlib.rcParams['savefig.dpi']             = 300
matplotlib.rcParams['font.size']               = 10
matplotlib.rcParams['figure.max_open_warning'] = 0
sns.set_style("whitegrid")

# ====================== 配置参数 ======================
CONFIG = {
    'data_file': (
        r"D:\博一下\第一章主动学习"
        r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
    ),
    'pkl_file': (
        r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
        r"\top30_models\model_objects"
        r"\rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl"
    ),
    'output_dir': r"D:\博二上\模型分析可视化\pdp4",

    'selected_features': [
        'deltaP_热导率W/(mk)',
        'x',
        'Ec',
        'Ω',
        'deltaP_E13 Electron affinity',
        'ΔSmix',
    ],
    'target_col':   'KQ',
    'random_state': 2023,

    # PDP参数
    'grid_resolution':    200,
    'percentiles':        (0.01, 0.99),
    'figure_size':        (12, 8),
    'dpi':                300,
    'histogram_bins':     50,

    # Bootstrap
    'bootstrap_samples':        1000,
    'bootstrap_samples_small':  2000,
    'bootstrap_samples_large':  500,
    'confidence_level':         0.95,
    'small_dataset_threshold':  1000,
    'large_dataset_threshold':  10000,

    # 交互分析
    'interaction_2d_resolution': 50,
    'interaction_3d_resolution': 30,
    'max_3d_interactions':       10,

    # ✅ 异常值检测：禁用（202个样本已是高质量筛选数据，无需再清洗）
    'outlier_detection': {
        'enable': False,
    }
}

# ====================== 工具函数 ======================
def cleanup_matplotlib():
    plt.close('all')
    import gc; gc.collect()

def safe_filename(name):
    replacements = {
        ' ': '_', '/': '_', '\\': '_', ':': '_', '*': '_', '?': '_',
        '"': '_', '<': '_', '>': '_', '|': '_', '(': '',  ')': '',
        '[': '',  ']': '',  '{': '',  '}': '',  '×': 'x', '÷': 'div',
        '+': 'plus', '=': 'eq', '%': 'percent', '#': 'num', '&': 'and',
        '@': 'at', '$': 'dollar', '!': 'excl', '^': 'caret', '~': 'tilde',
        '`': 'grave', "'": '', ',': '_', ';': '_', '.': '_', '-': '_',
        'Δ': 'Delta', 'δ': 'delta', 'Ω': 'Omega', 'Λ': 'Lambda',
        'α': 'alpha', 'β': 'beta', 'γ': 'gamma',
    }
    s = name
    for old, new in replacements.items():
        s = s.replace(old, new)
    while '__' in s:
        s = s.replace('__', '_')
    s = s.strip('_')
    return s[:50]

def get_adaptive_config(n_samples):
    if n_samples <= CONFIG['small_dataset_threshold']:
        return {'bootstrap_samples': CONFIG['bootstrap_samples_small']}
    elif n_samples <= CONFIG['large_dataset_threshold']:
        return {'bootstrap_samples': CONFIG['bootstrap_samples']}
    else:
        return {'bootstrap_samples': CONFIG['bootstrap_samples_large']}

def calculate_histogram_data(feature_data, bins=None):
    if bins is None:
        bins = CONFIG['histogram_bins']
    counts, bin_edges = np.histogram(feature_data, bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width   = bin_edges[1] - bin_edges[0]
    densities   = counts / (len(feature_data) * bin_width)
    rel_freq    = counts / len(feature_data) * 100
    return {
        'bin_centers':          bin_centers,
        'bin_edges':            bin_edges,
        'counts':               counts,
        'densities':            densities,
        'relative_frequencies': rel_freq,
        'bin_width':            bin_width,
        'total_samples':        len(feature_data)
    }

def save_data_to_excel(data, filepath, sheet_name='Data', additional_info=None):
    try:
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            if isinstance(data, dict):
                for key, value in data.items():
                    sname = safe_filename(key)[:31]
                    if isinstance(value, np.ndarray):
                        if value.ndim == 1:
                            pd.DataFrame(
                                {safe_filename(key): value}
                            ).to_excel(writer, sheet_name=sname, index=False)
                        else:
                            pd.DataFrame(value).to_excel(
                                writer, sheet_name=sname, index=True)
                    elif isinstance(value, pd.DataFrame):
                        value.to_excel(writer, sheet_name=sname, index=False)
                    elif isinstance(value, (list, tuple)):
                        try:
                            pd.DataFrame(
                                {safe_filename(key): value}
                            ).to_excel(writer, sheet_name=sname, index=False)
                        except:
                            pd.DataFrame(
                                {'data': [str(v) for v in value]}
                            ).to_excel(writer, sheet_name=sname, index=False)
                    else:
                        pd.DataFrame(
                            {safe_filename(key): [str(value)]}
                        ).to_excel(writer, sheet_name=sname, index=False)
            elif isinstance(data, pd.DataFrame):
                data.to_excel(
                    writer,
                    sheet_name=safe_filename(sheet_name)[:31],
                    index=False)
            elif isinstance(data, (list, np.ndarray)):
                sname = safe_filename(sheet_name)[:31]
                if isinstance(data, np.ndarray) and data.ndim > 1:
                    pd.DataFrame(data).to_excel(
                        writer, sheet_name=sname, index=True)
                else:
                    pd.DataFrame(
                        {safe_filename(sheet_name): data}
                    ).to_excel(writer, sheet_name=sname, index=False)
            if additional_info:
                try:
                    pd.DataFrame(
                        list(additional_info.items()), columns=['项目', '值']
                    ).to_excel(writer, sheet_name='Info', index=False)
                except:
                    pass
        return True
    except Exception as e:
        print(f"      ❌ Excel保存失败: {e}")
        return False

def save_2d_interaction_data(f1v, f2v, Z, f1n, f2n, excel_path, info=None):
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            pd.DataFrame({'feature1_values': f1v}).to_excel(
                writer, sheet_name='Feature1_Values', index=False)
            pd.DataFrame({'feature2_values': f2v}).to_excel(
                writer, sheet_name='Feature2_Values', index=False)
            pd.DataFrame(Z).to_excel(
                writer, sheet_name='PDP_Matrix', index=True)
            XX, YY = np.meshgrid(f1v, f2v)
            long = [
                {'f1': XX[ii, jj], 'f2': YY[ii, jj], 'pdp': Z[ii, jj]}
                for ii in range(Z.shape[0])
                for jj in range(Z.shape[1])
            ]
            pd.DataFrame(long).to_excel(
                writer, sheet_name='Long_Format', index=False)
            if info:
                pd.DataFrame(
                    list(info.items()), columns=['项目', '值']
                ).to_excel(writer, sheet_name='Info', index=False)
        return True
    except Exception as e:
        print(f"      ❌ 2D Excel保存失败: {e}")
        return False

def save_3d_interaction_data(f1v, f2v, f3v, Z3d,
                              f1n, f2n, f3n, excel_path, info=None):
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            pd.DataFrame({'feature1_values': f1v}).to_excel(
                writer, sheet_name='Feature1_Values', index=False)
            pd.DataFrame({'feature2_values': f2v}).to_excel(
                writer, sheet_name='Feature2_Values', index=False)
            pd.DataFrame({'feature3_values': f3v}).to_excel(
                writer, sheet_name='Feature3_Values', index=False)
            flat = [
                {'f1': v1, 'f2': v2, 'f3': v3, 'pdp': Z3d[i, j, k]}
                for i, v1 in enumerate(f1v)
                for j, v2 in enumerate(f2v)
                for k, v3 in enumerate(f3v)
            ]
            pd.DataFrame(flat).to_excel(
                writer, sheet_name='3D_Flat', index=False)
            for sidx in [0, len(f3v) // 2, len(f3v) - 1]:
                df_sl = pd.DataFrame(Z3d[:, :, sidx])
                sname = f'Slice_F3_{f3v[sidx]:.3f}'[:31]
                df_sl.to_excel(writer, sheet_name=sname, index=True)
            if info:
                pd.DataFrame(
                    list(info.items()), columns=['项目', '值']
                ).to_excel(writer, sheet_name='Info', index=False)
        return True
    except Exception as e:
        print(f"      ❌ 3D Excel保存失败: {e}")
        return False

# ====================== GP专用预测函数 ======================
def gp_predict_scaled(model, scaler, X_raw):
    """GP预测：先标准化再预测"""
    return model.predict(scaler.transform(X_raw))

# ====================== 特征重要性（适配GP模型）======================
def calculate_feature_importance(model, X_data, y_data, feature_names, scaler):
    """
    GP模型没有feature_importances_，使用置换重要性。
    ScaledPredictor 拥有 predict / score / fit / __sklearn_is_fitted__ 方法。
    """
    print("    计算置换重要性（高斯过程模型专用）...")

    class ScaledPredictor:
        def __init__(self, gp_model, sc):
            self.gp_model = gp_model
            self.sc       = sc

        def predict(self, X):
            return self.gp_model.predict(self.sc.transform(X))

        def score(self, X, y):
            return r2_score(y, self.predict(X))

        def fit(self, X, y=None):
            return self

        def __sklearn_is_fitted__(self):
            return True

    wrapped = ScaledPredictor(model, scaler)
    perm = permutation_importance(
        wrapped, X_data, y_data,
        n_repeats=10,
        random_state=CONFIG['random_state'],
        n_jobs=1
    )
    importances = np.maximum(perm.importances_mean, 0)
    return importances, "Permutation Importance"

# ====================== 加载模型和数据 ======================
def load_gp_model_and_data():
    print("=== 加载TOP1高斯过程模型和数据 ===")
    with open(CONFIG['pkl_file'], 'rb') as f:
        model_data = pickle.load(f)
    gp_model      = model_data['model']
    scaler        = model_data['scaler']
    feature_names = model_data['features']
    print(f"✓ 模型类型: {model_data['model_name']}")
    print(f"✓ 特征列表: {feature_names}")

    df = pd.read_excel(CONFIG['data_file'])
    print(f"✓ 原始数据: {df.shape}")

    df_valid = df[feature_names + [CONFIG['target_col']]].dropna()
    df_valid = df_valid[
        df_valid[CONFIG['target_col']] > 0].reset_index(drop=True)
    X_all = df_valid[feature_names].values
    y_all = df_valid[CONFIG['target_col']].values
    print(f"✓ 有效样本: {len(df_valid)}")

    from sklearn.model_selection import train_test_split

    def create_stratify_bins(y, n_bins=5):
        percentiles = np.linspace(0, 100, n_bins + 1)
        bin_edges   = np.percentile(y, percentiles)
        bin_edges[0]  = -np.inf
        bin_edges[-1] =  np.inf
        return np.digitize(y, bin_edges) - 1

    strat   = create_stratify_bins(y_all)
    idx_all = np.arange(len(y_all))
    train_idx, test_idx, _, _ = train_test_split(
        idx_all, idx_all, test_size=0.2,
        stratify=strat, random_state=2023)
    print(f"✓ 训练集: {len(train_idx)}  测试集: {len(test_idx)}")

    return {
        'model_name':     model_data['model_name'],
        'model_instance': gp_model,
        'scaler':         scaler,
        'features':       feature_names,
        'X_data':         X_all,
        'y_data':         y_all,
        'train_idx':      train_idx,
        'test_idx':       test_idx,
        'test_r2':        model_data.get('test_r2_true',  np.nan),
        'test_mae':       model_data.get('test_mae_true', np.nan),
    }

# ====================== 创建输出目录 ======================
def create_output_dirs(base_dir):
    for s in ['combined_pdp', 'interaction_2d_pdp',
              'interaction_3d_pdp', 'outlier_detection', 'cleaned_analysis']:
        os.makedirs(os.path.join(base_dir, s), exist_ok=True)
    print(f"✓ 输出目录已创建: {base_dir}")

# ====================== 1D PDP 分析 ======================
def generate_combined_pdp_analysis(model_info, out_dir):
    print(f"\n  生成1D PDP分析（Individual + Bootstrap置信区间）...")
    model    = model_info['model_instance']
    scaler   = model_info['scaler']
    X_data   = model_info['X_data_cleaned']
    y_data   = model_info['y_data_cleaned']
    features = model_info['features']

    adaptive = get_adaptive_config(X_data.shape[0])
    n_boot   = adaptive['bootstrap_samples']
    print(f"    样本数: {X_data.shape[0]}  "
          f"Bootstrap: {n_boot}次  "
          f"网格: {CONFIG['grid_resolution']}")

    importances, imp_type = calculate_feature_importance(
        model, X_data, y_data, features, scaler)

    combined_data_all = {}

    for i, feature in enumerate(features):
        print(f"    处理特征: {feature}")
        try:
            feat_data  = X_data[:, i]
            hist_data  = calculate_histogram_data(feat_data)
            lo = np.percentile(feat_data, CONFIG['percentiles'][0] * 100)
            hi = np.percentile(feat_data, CONFIG['percentiles'][1] * 100)
            fixed_grid = np.linspace(lo, hi, CONFIG['grid_resolution'])

            # 基础PDP（手动计算）
            base_pdp = []
            for g in fixed_grid:
                Xm = X_data.copy(); Xm[:, i] = g
                base_pdp.append(
                    np.mean(gp_predict_scaled(model, scaler, Xm)))
            base_pdp = np.array(base_pdp)

            # Bootstrap
            print(f"      Bootstrap {n_boot} 次...")
            boot_pdps = []
            for b in range(n_boot):
                try:
                    bidx = np.random.choice(
                        X_data.shape[0], X_data.shape[0], replace=True)
                    Xb   = X_data[bidx]
                    bpdp = []
                    for g in fixed_grid:
                        Xm = Xb.copy(); Xm[:, i] = g
                        bpdp.append(
                            np.mean(gp_predict_scaled(model, scaler, Xm)))
                    boot_pdps.append(bpdp)
                    if (b + 1) % 200 == 0:
                        print(f"        进度: {b + 1}/{n_boot}")
                except:
                    continue

            # 置信区间
            cis = {}
            if len(boot_pdps) >= 10:
                boot_arr = np.array(boot_pdps)
                for cl in [0.90, 0.95, 0.99]:
                    alpha = 1 - cl
                    lo_p  = alpha / 2 * 100
                    hi_p  = (1 - alpha / 2) * 100
                    cis[f'{int(cl * 100)}%'] = {
                        'lower': np.percentile(boot_arr, lo_p,  axis=0),
                        'upper': np.percentile(boot_arr, hi_p,  axis=0),
                        'width': np.percentile(boot_arr, hi_p,  axis=0) -
                                 np.percentile(boot_arr, lo_p,  axis=0)
                    }
                boot_mean    = np.mean(boot_arr, axis=0)
                boot_std     = np.std(boot_arr,  axis=0)
                main_key     = f'{int(CONFIG["confidence_level"] * 100)}%'
                ci_lower     = cis[main_key]['lower']
                ci_upper     = cis[main_key]['upper']
                success_rate = len(boot_pdps) / n_boot * 100
            else:
                boot_mean = boot_std = ci_lower = ci_upper = None
                success_rate = 0

            # 绘图（3子图）
            fig, axes = plt.subplots(
                3, 1,
                figsize=(CONFIG['figure_size'][0],
                         CONFIG['figure_size'][1] * 1.5))

            ax1    = axes[0]
            colors = ['lightblue', 'lightgreen', 'lightcoral']
            if ci_lower is not None:
                for idx_ci, (level, ci) in enumerate(cis.items()):
                    ax1.fill_between(
                        fixed_grid, ci['lower'], ci['upper'],
                        alpha=max(0.1, 0.5 - idx_ci * 0.1),
                        color=colors[idx_ci], label=f'{level} CI')
                ax1.plot(fixed_grid, boot_mean, 'r--', lw=2,
                         alpha=0.8, label='Bootstrap均值')
            ax1.plot(fixed_grid, base_pdp, 'blue', lw=4,
                     label='PDP曲线', zorder=10)
            ax1t = ax1.twinx()
            ax1t.hist(feat_data, bins=CONFIG['histogram_bins'],
                      alpha=0.3, color='gray', density=True)
            ax1t.set_ylabel('特征密度', color='gray', fontsize=10)
            ax1t.tick_params(axis='y', labelcolor='gray')
            ax1.set_xlabel(feature, fontsize=12, fontweight='bold')
            ax1.set_ylabel('偏依赖值', fontsize=12, fontweight='bold')
            ax1.set_title(
                f'PDP分析: {feature}\n'
                f'重要性: {importances[i]:.4f} ({imp_type})\n'
                f'Bootstrap: {len(boot_pdps)}次  '
                f'成功率: {success_rate:.1f}%',
                fontsize=12, fontweight='bold')
            ax1.legend(loc='upper left'); ax1.grid(alpha=0.3)

            ax2      = axes[1]
            pdp_grad = np.gradient(base_pdp, fixed_grid)
            ax2.plot(fixed_grid, pdp_grad, 'red', lw=3, label='PDP斜率')
            ax2.axhline(0, color='k', ls='--', alpha=0.5)
            ax2.set_xlabel(feature, fontsize=10, fontweight='bold')
            ax2.set_ylabel('PDP斜率', fontsize=10, fontweight='bold')
            ax2.set_title('特征效应变化率')
            ax2.legend(); ax2.grid(alpha=0.3)
            max_grad = np.max(np.abs(pdp_grad))

            ax3 = axes[2]
            if ci_lower is not None:
                ci_width = ci_upper - ci_lower
                ax3.plot(fixed_grid, ci_width,  'purple', lw=3,
                         label='置信区间宽度')
                ax3.plot(fixed_grid, boot_std,  'orange', lw=3,
                         label='Bootstrap std')
                ax3.set_xlabel(feature, fontsize=10, fontweight='bold')
                ax3.set_ylabel('不确定性', fontsize=10, fontweight='bold')
                ax3.set_title('不确定性分析')
                ax3.legend(); ax3.grid(alpha=0.3)
                avg_ci  = np.mean(ci_width)
            else:
                ax3.text(0.5, 0.5, 'Bootstrap不足\n无法分析不确定性',
                         ha='center', va='center', transform=ax3.transAxes)
                avg_ci = 0

            stats_txt = (
                f'网格: {CONFIG["grid_resolution"]}点\n'
                f'样本数: {len(feat_data)}\n'
                f'均值: {np.mean(feat_data):.4f}\n'
                f'std: {np.std(feat_data):.4f}\n'
                f'PDP幅度: {np.ptp(base_pdp):.4f}\n'
                f'最大斜率: {max_grad:.4f}\n'
                f'平均不确定性: {avg_ci:.4f}'
            )
            ax1.text(
                0.02, 0.98, stats_txt, transform=ax1.transAxes,
                bbox=dict(boxstyle='round,pad=0.3', fc='lightcyan', alpha=0.8),
                va='top', fontsize=9)

            plt.tight_layout()
            sname = safe_filename(feature)
            plt.savefig(
                os.path.join(out_dir, 'combined_pdp',
                             f'PDP_Analysis_{sname}.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight')
            plt.close()

            # 保存Excel
            excel_data = {
                'feature_values':   fixed_grid,
                'base_pdp':         base_pdp,
                'pdp_gradient':     pdp_grad,
                'feature_data':     feat_data,
                'hist_bin_centers': hist_data['bin_centers'],
                'hist_counts':      hist_data['counts'],
                'hist_densities':   hist_data['densities'],
            }
            if ci_lower is not None:
                excel_data.update({
                    'bootstrap_mean': boot_mean,
                    'bootstrap_std':  boot_std,
                    'ci_width':       ci_width,
                })
                for level, ci in cis.items():
                    excel_data[f'ci_{level}_lower'] = ci['lower']
                    excel_data[f'ci_{level}_upper'] = ci['upper']
                    excel_data[f'ci_{level}_width'] = ci['width']

            save_data_to_excel(
                excel_data,
                os.path.join(out_dir, 'combined_pdp',
                             f'PDP_Data_{sname}.xlsx'),
                'PDP数据',
                {'特征': feature, '重要性': importances[i],
                 '样本数': len(feat_data), 'Bootstrap次数': len(boot_pdps)})

            combined_data_all[feature] = {
                'feature_values':       fixed_grid,
                'base_pdp':             base_pdp,
                'feature_importance':   importances[i],
                'ci_lower':             ci_lower,
                'ci_upper':             ci_upper,
                'cleaned_feature_data': feat_data,
                'histogram_data':       hist_data,
            }
            print(f"      ✓ {feature} PDP分析完成")

        except Exception as e:
            print(f"      ❌ {feature} PDP分析失败: {e}")
            import traceback; traceback.print_exc()
            plt.close('all')
            continue

    # 综合对比图
    if combined_data_all:
        try:
            n_f  = len(combined_data_all)
            cols = min(3, n_f)
            rows = (n_f + cols - 1) // cols
            fig, axes_arr = plt.subplots(rows, cols,
                                         figsize=(6 * cols, 5 * rows))
            if n_f == 1:
                axes_arr = np.array([[axes_arr]])
            elif rows == 1:
                axes_arr = axes_arr.reshape(1, -1)

            for idx, (feat, fdata) in enumerate(combined_data_all.items()):
                r, c = divmod(idx, cols)
                ax   = axes_arr[r, c]
                fv   = fdata['feature_values']
                bp   = fdata['base_pdp']
                if fdata['ci_lower'] is not None:
                    ax.fill_between(fv, fdata['ci_lower'], fdata['ci_upper'],
                                    alpha=0.3, color='lightblue', label='95% CI')
                ax.plot(fv, bp, 'blue', lw=2, label='PDP')
                axt = ax.twinx()
                axt.hist(fdata['cleaned_feature_data'], bins=20,
                         alpha=0.2, color='gray', density=True)
                axt.set_ylabel('密度', fontsize=7, color='gray')
                ax.set_xlabel(feat, fontsize=9)
                ax.set_ylabel('PDP', fontsize=9)
                ax.set_title(
                    f'{feat}\n重要性: {fdata["feature_importance"]:.4f}',
                    fontsize=9, fontweight='bold')
                ax.legend(fontsize=7); ax.grid(alpha=0.3)

            for idx in range(n_f, rows * cols):
                r, c = divmod(idx, cols)
                axes_arr[r, c].axis('off')

            plt.suptitle('所有特征PDP综合对比（全部202个样本）',
                         fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(
                os.path.join(out_dir, 'combined_pdp',
                             'All_Features_PDP_Comprehensive.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight')
            plt.close()
            print("    ✓ PDP综合对比图已保存")
        except Exception as e:
            print(f"    ❌ 综合对比图失败: {e}")
            plt.close('all')

    return combined_data_all

# ====================== ✅ 2D交互PDP（完全手动计算）======================
def generate_complete_2d_interaction_pdp(model_info, out_dir):
    print(f"\n  生成2D特征交互PDP分析（所有特征对）...")
    model    = model_info['model_instance']
    scaler   = model_info['scaler']
    X_data   = model_info['X_data_cleaned']
    features = model_info['features']

    combos = list(combinations(range(len(features)), 2))
    print(f"    特征对总数: {len(combos)}")
    print(f"    2D网格: "
          f"{CONFIG['interaction_2d_resolution']}×"
          f"{CONFIG['interaction_2d_resolution']}")
    print(f"    ✅ 完全手动计算，不依赖 partial_dependence")

    interaction_data = {}

    for cidx, (i, j) in enumerate(combos):
        fn1 = features[i]; fn2 = features[j]
        print(f"    [{cidx + 1}/{len(combos)}] {fn1} × {fn2}")
        try:
            # 计算特征范围
            f1_lo = np.percentile(X_data[:, i],
                                  CONFIG['percentiles'][0] * 100)
            f1_hi = np.percentile(X_data[:, i],
                                  CONFIG['percentiles'][1] * 100)
            f2_lo = np.percentile(X_data[:, j],
                                  CONFIG['percentiles'][0] * 100)
            f2_hi = np.percentile(X_data[:, j],
                                  CONFIG['percentiles'][1] * 100)

            res = CONFIG['interaction_2d_resolution']
            f1v = np.linspace(f1_lo, f1_hi, res)
            f2v = np.linspace(f2_lo, f2_hi, res)

            # ✅ 手动计算2D PDP网格
            # Z[jj, ii] 对应 f2v[jj] × f1v[ii]
            Z = np.zeros((len(f2v), len(f1v)))
            for ii, v1 in enumerate(f1v):
                for jj, v2 in enumerate(f2v):
                    Xm = X_data.copy()
                    Xm[:, i] = v1
                    Xm[:, j] = v2
                    Z[jj, ii] = np.mean(
                        gp_predict_scaled(model, scaler, Xm))
                if (ii + 1) % 10 == 0:
                    print(f"      进度: {ii + 1}/{res}")

            XX, YY = np.meshgrid(f1v, f2v)

            # 绘图：热力图 + 3D表面图
            from mpl_toolkits.mplot3d import Axes3D
            fig = plt.figure(
                figsize=(CONFIG['figure_size'][0] * 1.5,
                         CONFIG['figure_size'][1]))

            ax1 = fig.add_subplot(121)
            im  = ax1.contourf(XX, YY, Z, levels=50, cmap='RdYlBu_r')
            fig.colorbar(im, ax=ax1, label='偏依赖值')
            ct  = ax1.contour(XX, YY, Z, levels=10,
                              colors='k', alpha=0.6, linewidths=0.8)
            ax1.clabel(ct, inline=True, fontsize=8, fmt='%.3f')
            ax1.scatter(X_data[:, i], X_data[:, j],
                        c='white', s=0.5, alpha=0.3)
            ax1.set_xlabel(fn1, fontsize=11, fontweight='bold')
            ax1.set_ylabel(fn2, fontsize=11, fontweight='bold')
            ax1.set_title(f'2D热力图\n{fn1} × {fn2}',
                          fontsize=11, fontweight='bold')

            ax2  = fig.add_subplot(122, projection='3d')
            surf = ax2.plot_surface(XX, YY, Z, cmap='RdYlBu_r', alpha=0.9)
            fig.colorbar(surf, ax=ax2, label='偏依赖值', shrink=0.5)
            ax2.set_xlabel(fn1, fontsize=9)
            ax2.set_ylabel(fn2, fontsize=9)
            ax2.set_zlabel('偏依赖值', fontsize=9)
            ax2.set_title(f'3D表面图\n{fn1} × {fn2}',
                          fontsize=9, fontweight='bold')

            plt.tight_layout()
            sn1 = safe_filename(fn1); sn2 = safe_filename(fn2)
            plt.savefig(
                os.path.join(out_dir, 'interaction_2d_pdp',
                             f'2D_Interaction_{sn1}_{sn2}.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight')
            plt.close()

            interaction_data[f"{fn1}_×_{fn2}"] = {
                'f1v': f1v, 'f2v': f2v, 'Z': Z,
                'f1':  fn1, 'f2':  fn2,
                'interaction_strength': np.var(Z)
            }
            save_2d_interaction_data(
                f1v, f2v, Z, fn1, fn2,
                os.path.join(out_dir, 'interaction_2d_pdp',
                             f'2D_Interaction_Data_{sn1}_{sn2}.xlsx'),
                {'特征1': fn1, '特征2': fn2,
                 '交互强度': np.var(Z), '样本数': X_data.shape[0]})
            print(f"      ✓ 完成")

        except Exception as e:
            print(f"      ❌ 失败: {e}")
            import traceback; traceback.print_exc()
            plt.close('all')

    # 交互强度排序
    if interaction_data:
        try:
            sorted_int = sorted(
                interaction_data.items(),
                key=lambda x: x[1]['interaction_strength'],
                reverse=True)
            top10 = sorted_int[:min(10, len(sorted_int))]
            names = [n.replace('_×_', ' × ') for n, _ in top10]
            strs  = [d['interaction_strength'] for _, d in top10]

            fig, ax = plt.subplots(figsize=(14, 6))
            bars = ax.bar(
                range(len(names)), strs,
                color=plt.cm.viridis(np.linspace(0, 1, len(names))))
            ax.set_xticks(range(len(names)))
            ax.set_xticklabels(names, rotation=45, ha='right')
            ax.set_xlabel('特征交互对', fontsize=12, fontweight='bold')
            ax.set_ylabel('交互强度 (PDP方差)',
                          fontsize=12, fontweight='bold')
            ax.set_title(f'Top{len(top10)} 2D特征交互强度排序',
                         fontsize=14, fontweight='bold')
            for bar, s in zip(bars, strs):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(strs) * 0.01,
                    f'{s:.4f}', ha='center', va='bottom', fontsize=9)
            ax.grid(alpha=0.3, axis='y')
            plt.tight_layout()
            plt.savefig(
                os.path.join(out_dir, 'interaction_2d_pdp',
                             'Interaction_Strength_Ranking.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight')
            plt.close()

            pd.DataFrame([
                {'排名': r+1, '交互对': n.replace('_×_', ' × '),
                 '特征1': d['f1'], '特征2': d['f2'],
                 '交互强度': d['interaction_strength'],
                 'PDP幅度': np.ptp(d['Z'])}
                for r, (n, d) in enumerate(sorted_int)
            ]).to_excel(
                os.path.join(out_dir, 'interaction_2d_pdp',
                             'All_2D_Interactions_Ranking.xlsx'),
                index=False)
            print("    ✓ 交互强度排序报告已保存")
        except Exception as e:
            print(f"    ❌ 排序报告失败: {e}")
            plt.close('all')

    print(f"    ✓ 2D分析完成：{len(interaction_data)}/{len(combos)}")
    return interaction_data

# ====================== 3D交互PDP ======================
def generate_selected_3d_interaction_pdp(model_info, out_dir):
    print(f"\n  生成3D特征交互PDP分析...")
    model    = model_info['model_instance']
    scaler   = model_info['scaler']
    X_data   = model_info['X_data_cleaned']
    y_data   = model_info['y_data_cleaned']
    features = model_info['features']

    importances, _ = calculate_feature_importance(
        model, X_data, y_data, features, scaler)
    top_idx  = np.argsort(importances)[-min(6, len(features)):]
    combos3d = list(combinations(top_idx, 3))
    if len(combos3d) > CONFIG['max_3d_interactions']:
        combos3d = combos3d[:CONFIG['max_3d_interactions']]

    print(f"    Top{len(top_idx)}重要特征，共{len(combos3d)}个3D组合")
    print(f"    3D网格: {CONFIG['interaction_3d_resolution']}³")

    if not combos3d:
        print("    ⚠️ 没有足够特征进行3D分析")
        return {}

    interaction_3d_data = {}

    for cidx, (i, j, k) in enumerate(combos3d):
        fn1 = features[i]; fn2 = features[j]; fn3 = features[k]
        print(f"    [{cidx + 1}/{len(combos3d)}] "
              f"{fn1} × {fn2} × {fn3}")
        try:
            r1 = np.linspace(
                np.percentile(X_data[:, i], 1),
                np.percentile(X_data[:, i], 99),
                CONFIG['interaction_3d_resolution'])
            r2 = np.linspace(
                np.percentile(X_data[:, j], 1),
                np.percentile(X_data[:, j], 99),
                CONFIG['interaction_3d_resolution'])
            r3 = np.linspace(
                np.percentile(X_data[:, k], 1),
                np.percentile(X_data[:, k], 99),
                CONFIG['interaction_3d_resolution'])

            Z3d   = np.zeros((len(r1), len(r2), len(r3)))
            total = len(r1) * len(r2) * len(r3)
            print(f"      计算 {total} 个网格点...")

            for ii, v1 in enumerate(r1):
                for jj, v2 in enumerate(r2):
                    for kk, v3 in enumerate(r3):
                        Xm = X_data.copy()
                        Xm[:, i] = v1
                        Xm[:, j] = v2
                        Xm[:, k] = v3
                        Z3d[ii, jj, kk] = np.mean(
                            gp_predict_scaled(model, scaler, Xm))
                if (ii + 1) % 5 == 0:
                    pct = (ii + 1) * len(r2) * len(r3) / total * 100
                    print(f"        进度: {pct:.1f}%")

            # 绘图（6切片）
            from mpl_toolkits.mplot3d import Axes3D
            fig, axes_arr = plt.subplots(2, 3, figsize=(18, 12))
            axes_arr = axes_arr.ravel()
            slices   = [0, len(r3) // 4, len(r3) // 2,
                        3 * len(r3) // 4, len(r3) - 1]
            slices   = slices[:6]

            for sidx, ax in enumerate(axes_arr):
                if sidx < len(slices):
                    f3i  = slices[sidx]
                    f3v  = r3[f3i]
                    XX2, YY2 = np.meshgrid(r2, r1)
                    Zsl  = Z3d[:, :, f3i]
                    im   = ax.contourf(XX2, YY2, Zsl,
                                       levels=20, cmap='RdYlBu_r')
                    fig.colorbar(im, ax=ax)
                    ax.contour(XX2, YY2, Zsl, levels=8,
                               colors='k', alpha=0.5, linewidths=0.5)
                    ax.set_xlabel(fn2, fontsize=9)
                    ax.set_ylabel(fn1, fontsize=9)
                    ax.set_title(f'{fn3}={f3v:.3f}',
                                 fontsize=9, fontweight='bold')
                else:
                    ax.axis('off')

            plt.suptitle(
                f'3D交互PDP切片: {fn1} × {fn2} × {fn3}',
                fontsize=14, fontweight='bold')
            plt.tight_layout()
            sn1 = safe_filename(fn1)
            sn2 = safe_filename(fn2)
            sn3 = safe_filename(fn3)
            plt.savefig(
                os.path.join(out_dir, 'interaction_3d_pdp',
                             f'3D_Interaction_{sn1}_{sn2}_{sn3}.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight')
            plt.close()

            interaction_3d_data[f"{fn1}_×_{fn2}_×_{fn3}"] = {
                'r1': r1, 'r2': r2, 'r3': r3, 'Z3d': Z3d,
                'f1': fn1, 'f2': fn2, 'f3': fn3,
                'interaction_strength_3d': np.var(Z3d)
            }
            save_3d_interaction_data(
                r1, r2, r3, Z3d, fn1, fn2, fn3,
                os.path.join(out_dir, 'interaction_3d_pdp',
                             f'3D_Interaction_Data_{sn1}_{sn2}_{sn3}.xlsx'),
                {'特征1': fn1, '特征2': fn2, '特征3': fn3,
                 '3D交互强度': np.var(Z3d), '样本数': X_data.shape[0]})
            print(f"      ✓ 完成")

        except Exception as e:
            print(f"      ❌ 失败: {e}")
            import traceback; traceback.print_exc()
            plt.close('all')

    # 3D强度排序
    if interaction_3d_data:
        try:
            sorted3d = sorted(
                interaction_3d_data.items(),
                key=lambda x: x[1]['interaction_strength_3d'],
                reverse=True)
            names3d = [n.replace('_×_', ' × ') for n, _ in sorted3d]
            strs3d  = [d['interaction_strength_3d'] for _, d in sorted3d]

            fig, ax = plt.subplots(figsize=(12, 6))
            bars = ax.bar(
                range(len(names3d)), strs3d,
                color=plt.cm.plasma(np.linspace(0, 1, len(names3d))))
            ax.set_xticks(range(len(names3d)))
            ax.set_xticklabels(names3d, rotation=45, ha='right')
            ax.set_xlabel('3D特征交互组合', fontsize=12, fontweight='bold')
            ax.set_ylabel('3D交互强度', fontsize=12, fontweight='bold')
            ax.set_title('3D特征交互强度排序',
                         fontsize=14, fontweight='bold')
            for bar, s in zip(bars, strs3d):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() +
                    (max(strs3d) * 0.01 if strs3d else 0),
                    f'{s:.4f}', ha='center', va='bottom', fontsize=9)
            ax.grid(alpha=0.3, axis='y')
            plt.tight_layout()
            plt.savefig(
                os.path.join(out_dir, 'interaction_3d_pdp',
                             '3D_Interaction_Strength_Ranking.png'),
                dpi=CONFIG['dpi'], bbox_inches='tight')
            plt.close()

            pd.DataFrame([
                {'排名': r+1, '组合': n.replace('_×_', ' × '),
                 '特征1': d['f1'], '特征2': d['f2'], '特征3': d['f3'],
                 '3D交互强度': d['interaction_strength_3d'],
                 'PDP幅度': np.ptp(d['Z3d'])}
                for r, (n, d) in enumerate(sorted3d)
            ]).to_excel(
                os.path.join(out_dir, 'interaction_3d_pdp',
                             'All_3D_Interactions_Ranking.xlsx'),
                index=False)
            print("    ✓ 3D交互强度排序已保存")
        except Exception as e:
            print(f"    ❌ 3D排序失败: {e}")
            plt.close('all')

    print(f"    ✓ 3D分析完成：{len(interaction_3d_data)}/{len(combos3d)}")
    return interaction_3d_data

# ====================== 主分析函数 ======================
def analyze_gp_model():
    print("\n" + "=" * 80)
    print("TOP1 高斯过程模型 PDP 完整分析 v5")
    print("✅ 全部202个高质量样本直接用于PDP分析（异常值检测已禁用）")
    print("✅ 2D PDP 完全手动计算，彻底绕过 partial_dependence 类型检查")
    print("=" * 80)
    cleanup_matplotlib()

    out_dir = CONFIG['output_dir']
    create_output_dirs(out_dir)

    model_info = load_gp_model_and_data()
    X_orig     = model_info['X_data']
    y_orig     = model_info['y_data']
    features   = model_info['features']

    print(f"\n模型: {model_info['model_name']}")
    print(f"特征: {features}")
    print(f"样本数: {len(X_orig)}")
    print(f"Test R²: {model_info['test_r2']}")

    # 数据准备
    print(f"\n{'=' * 60}")
    print("1️⃣  数据准备")
    print('=' * 60)
    print("  ✅ 异常值检测已禁用")
    print(f"  ✅ 直接使用全部 {len(X_orig)} 个高质量样本")
    model_info['X_data_cleaned'] = X_orig
    model_info['y_data_cleaned'] = y_orig
    model_info['removed_indices'] = np.array([])

    # 1D PDP
    print(f"\n{'=' * 60}")
    print("2️⃣  1D PDP分析（Individual + Bootstrap置信区间）")
    print('=' * 60)
    combined_data = generate_combined_pdp_analysis(model_info, out_dir)

    # 2D交互
    print(f"\n{'=' * 60}")
    print("3️⃣  2D特征交互PDP分析（所有特征对，手动计算）")
    print('=' * 60)
    interaction_2d = generate_complete_2d_interaction_pdp(model_info, out_dir)

    # 3D交互
    print(f"\n{'=' * 60}")
    print("4️⃣  3D特征交互PDP分析（重要特征组合）")
    print('=' * 60)
    interaction_3d = generate_selected_3d_interaction_pdp(model_info, out_dir)

    # 汇总
    print(f"\n{'=' * 80}")
    print("✅ PDP分析完成！")
    print(f"{'=' * 80}")
    print(f"  分析样本数       : {len(X_orig)}（全部高质量样本）")
    print(f"  1D PDP分析特征数 : {len(combined_data)}")
    print(f"  2D交互分析对数   : {len(interaction_2d)}")
    print(f"  3D交互分析组数   : {len(interaction_3d)}")
    print(f"  输出目录         : {out_dir}")
    print(f"\n  子目录：")
    print(f"    combined_pdp/        → 1D PDP图 + Excel数据")
    print(f"    interaction_2d_pdp/  → 2D交互热力图 + Excel数据")
    print(f"    interaction_3d_pdp/  → 3D交互切片图 + Excel数据")

    cleanup_matplotlib()
    return True

# ====================== 入口 ======================
if __name__ == "__main__":
    analyze_gp_model()


TOP1 高斯过程模型 PDP 完整分析 v5
✅ 全部202个高质量样本直接用于PDP分析（异常值检测已禁用）
✅ 2D PDP 完全手动计算，彻底绕过 partial_dependence 类型检查
✓ 输出目录已创建: D:\博二上\模型分析可视化\pdp4
=== 加载TOP1高斯过程模型和数据 ===
✓ 模型类型: GP_Matern_0.5
✓ 特征列表: ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
✓ 原始数据: (202, 209)
✓ 有效样本: 202
✓ 训练集: 161  测试集: 41

模型: GP_Matern_0.5
特征: ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
样本数: 202
Test R²: 0.759772415859425

1️⃣  数据准备
  ✅ 异常值检测已禁用
  ✅ 直接使用全部 202 个高质量样本

2️⃣  1D PDP分析（Individual + Bootstrap置信区间）

  生成1D PDP分析（Individual + Bootstrap置信区间）...
    样本数: 202  Bootstrap: 2000次  网格: 200
    计算置换重要性（高斯过程模型专用）...
    处理特征: deltaP_热导率W/(mk)
      Bootstrap 2000 次...
        进度: 200/2000
        进度: 400/2000
        进度: 600/2000
        进度: 800/2000
        进度: 1000/2000
        进度: 1200/2000
        进度: 1400/2000
        进度: 1600/2000
        进度: 1800/2000
        进度: 2000/2000
      ✓ deltaP_热导率W/(mk) PDP分析完成
    处理特征: x
      Bootstrap 2000 次...
       